# Vitara AI — Vision Preprocessing Pipeline

> **Dataset Contract Ref:** Dataset Vision — Food Images  
> **Target Output:** `data/vision/processed/`  
> **Runtime:** Google Colab T4 GPU

---

## Pipeline Overview

```
Google Drive (raw)
  └── data/vision/raw/
        ├── nasi_goreng/   (≥500 img)
        ├── ayam_bakar/    (≥500 img)
        └── ...            (≥10 classes)
            ↓
    [Validation]
            ↓
    [Resize 224×224]
            ↓
    [Normalize: ImageNet mean/std]
            ↓
    [Augmentation: train only]
            ↓
    [Split: 80/10/10]
            ↓
Google Drive (processed)
  └── data/vision/processed/
        ├── train/
        ├── val/
        └── test/
```

## 0. Setup & Install Dependencies

In [ ]:
# Verify GPU
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!pip install -q Pillow tqdm scikit-learn pandas matplotlib seaborn

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ──────────────────────────────────────────
# KONFIGURASI
# ──────────────────────────────────────────
GDRIVE_ROOT   = "/content/drive/MyDrive"          # root Google Drive
RAW_DIR       = os.path.join(GDRIVE_ROOT, "data/vision/raw")        # folder dataset contract
PROCESSED_DIR = os.path.join(GDRIVE_ROOT, "data/vision/processed")  # output folder
CALORIE_MAP   = os.path.join(RAW_DIR, "calorie_map.csv")            # metadata kalori
CLASSES_TXT   = os.path.join(RAW_DIR, "classes.txt")                # daftar kelas

# Split ratio
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10

# Image spec (dataset contract: min 224×224)
IMG_SIZE    = 224

# Seed untuk reproducibility
RANDOM_SEED = 42

print(f"RAW_DIR       : {RAW_DIR}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")
print(f"Split         : {TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")

## 2. Validasi Dataset

In [ ]:
import os
import pandas as pd
from PIL import Image
from collections import defaultdict

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
MIN_IMAGES_PER_CLASS = 500
MIN_CLASSES = 10
MIN_RESOLUTION = (224, 224)
MAX_FILE_SIZE_MB = 2.0

def validate_dataset(raw_dir):
    """
    Validasi dataset sesuai Vitara AI Dataset Contract:
    - Format: JPEG atau PNG
    - Ukuran file: max 2MB
    - Resolusi: min 224×224
    - Min 500 gambar per kelas
    - Min 10 kelas
    - Tidak ada duplikat antar kelas
    """
    print("="*60)
    print("DATASET CONTRACT VALIDATION")
    print("="*60)

    classes = sorted([
        d for d in os.listdir(raw_dir)
        if os.path.isdir(os.path.join(raw_dir, d))
    ])

    report = []
    all_filenames = defaultdict(list)  # untuk cek duplikat
    issues = []

    for cls in classes:
        cls_dir = os.path.join(raw_dir, cls)
        files = [
            f for f in os.listdir(cls_dir)
            if os.path.splitext(f)[1].lower() in VALID_EXTENSIONS
        ]
        invalid_fmt  = []
        oversize     = []
        underres     = []
        valid_count  = 0

        for fname in files:
            fpath = os.path.join(cls_dir, fname)
            ext = os.path.splitext(fname)[1].lower()

            # Format check
            if ext not in VALID_EXTENSIONS:
                invalid_fmt.append(fname)
                continue

            # File size check
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            if size_mb > MAX_FILE_SIZE_MB:
                oversize.append(fname)

            # Resolution check
            try:
                with Image.open(fpath) as img:
                    w, h = img.size
                    if w < MIN_RESOLUTION[0] or h < MIN_RESOLUTION[1]:
                        underres.append(fname)
            except Exception:
                issues.append(f"  ⚠️  Corrupted: {cls}/{fname}")
                continue

            all_filenames[fname].append(cls)
            valid_count += 1

        class_ok = valid_count >= MIN_IMAGES_PER_CLASS
        status = "✅" if class_ok else "❌"
        report.append({
            "class": cls,
            "valid_images": valid_count,
            "invalid_format": len(invalid_fmt),
            "oversize": len(oversize),
            "underres": len(underres),
            "status": status
        })

        if not class_ok:
            issues.append(f"  ❌ {cls}: hanya {valid_count} gambar valid (min {MIN_IMAGES_PER_CLASS})")

    df_report = pd.DataFrame(report)
    print(df_report.to_string(index=False))

    # Cek duplikat antar kelas
    print("\n🔍 Cross-class Duplicate Check:")
    dup_count = 0
    for fname, cls_list in all_filenames.items():
        if len(cls_list) > 1:
            issues.append(f"  ⚠️  Duplikat: '{fname}' ditemukan di {cls_list}")
            dup_count += 1
    print(f"  {'✅ Tidak ada duplikat antar kelas.' if dup_count == 0 else f'❌ {dup_count} duplikat ditemukan.'}")

    # Class count check
    print(f"\nJumlah Kelas : {len(classes)} {'✅' if len(classes) >= MIN_CLASSES else '❌ (min 10)'}")

    if issues:
        print("\n⚠️  Issues:")
        for i in issues:
            print(i)
    else:
        print("\n✅ Semua validasi passed!")

    return df_report, classes

df_report, classes = validate_dataset(RAW_DIR)
print(f"\nClasses: {classes}")

## 3. Validasi calorie_map.csv & classes.txt

In [ ]:
# ── calorie_map.csv ──────────────────────────────────
print("Validasi calorie_map.csv")
df_cal = pd.read_csv(CALORIE_MAP)
required_cols = {'class_name', 'calories_per_100g'}
assert required_cols.issubset(df_cal.columns), f"Missing columns: {required_cols - set(df_cal.columns)}"
assert df_cal['class_name'].notna().all(), "Ada null di class_name"
assert df_cal['calories_per_100g'].dtype in ['int64', 'float64'], "calories_per_100g harus numerik"

# Cek semua folder class ada di calorie_map
cal_classes = set(df_cal['class_name'].tolist())
folder_classes = set(classes)
missing_in_cal = folder_classes - cal_classes
extra_in_cal   = cal_classes - folder_classes

if missing_in_cal:
    print(f"  ⚠️  Kelas di folder tapi tidak di calorie_map: {missing_in_cal}")
if extra_in_cal:
    print(f"  ⚠️  Kelas di calorie_map tapi tidak ada foldernya: {extra_in_cal}")
if not missing_in_cal and not extra_in_cal:
    print("  ✅ calorie_map.csv lengkap dan sinkron dengan folder kelas.")

print(df_cal)

# ── classes.txt ──────────────────────────────────────
print("\nValidasi classes.txt")
with open(CLASSES_TXT) as f:
    txt_classes = [line.strip() for line in f if line.strip()]

if set(txt_classes) == folder_classes:
    print(f"  ✅ classes.txt match dengan folder ({len(txt_classes)} kelas).")
else:
    diff = set(txt_classes).symmetric_difference(folder_classes)
    print(f"  ⚠️  Mismatch: {diff}")

## 4. EDA — Distribusi Dataset Sebelum Split

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar plot: jumlah gambar per kelas
ax = axes[0]
colors = sns.color_palette("muted", len(df_report))
bars = ax.bar(df_report['class'], df_report['valid_images'], color=colors)
ax.axhline(500, color='red', linestyle='--', linewidth=1.2, label='Min 500 (contract)')
ax.set_xlabel('Kelas Makanan')
ax.set_ylabel('Jumlah Gambar Valid')
ax.set_title('Distribusi Gambar per Kelas')
ax.tick_params(axis='x', rotation=45)
ax.legend()
for bar, val in zip(bars, df_report['valid_images']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontsize=8)

# Bar plot: kalori per kelas
ax2 = axes[1]
merged = df_report.merge(df_cal, left_on='class', right_on='class_name', how='left')
ax2.bar(merged['class'], merged['calories_per_100g'],
        color=sns.color_palette("coolwarm", len(merged)))
ax2.set_xlabel('Kelas Makanan')
ax2.set_ylabel('Kalori per 100g (kkal)')
ax2.set_title('Estimasi Kalori per Kelas')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('/content/eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Plot disimpan: /content/eda_distribution.png")

## 5. Train/Val/Test Split

Split per kelas (stratified) agar distribusi label seimbang di semua subset.

In [ ]:
import random
from pathlib import Path

random.seed(RANDOM_SEED)

def collect_image_paths(raw_dir, classes, valid_exts=VALID_EXTENSIONS):
    """Return dict {class: [list of Path]}"""
    data = {}
    for cls in classes:
        cls_dir = Path(raw_dir) / cls
        imgs = sorted([
            p for p in cls_dir.iterdir()
            if p.suffix.lower() in valid_exts
        ])
        data[cls] = imgs
    return data

def stratified_split(data, train_ratio=0.8, val_ratio=0.1, seed=42):
    """
    Stratified split per kelas → (train, val, test) list of (path, label) tuples.
    """
    train_set, val_set, test_set = [], [], []

    for cls, paths in data.items():
        paths = paths.copy()
        random.Random(seed).shuffle(paths)

        n = len(paths)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)

        train_paths = paths[:n_train]
        val_paths   = paths[n_train:n_train + n_val]
        test_paths  = paths[n_train + n_val:]

        train_set += [(p, cls) for p in train_paths]
        val_set   += [(p, cls) for p in val_paths]
        test_set  += [(p, cls) for p in test_paths]

    return train_set, val_set, test_set

image_data = collect_image_paths(RAW_DIR, classes)
train_set, val_set, test_set = stratified_split(
    image_data, TRAIN_RATIO, VAL_RATIO, RANDOM_SEED
)

print(f"Split Summary")
print(f"  Train : {len(train_set):>6} gambar  ({100*TRAIN_RATIO:.0f}%)")
print(f"  Val   : {len(val_set):>6} gambar  ({100*VAL_RATIO:.0f}%)")
print(f"  Test  : {len(test_set):>6} gambar  ({100*TEST_RATIO:.0f}%)")
print(f"  Total : {len(train_set)+len(val_set)+len(test_set):>6} gambar")

In [ ]:
# Verifikasi distribusi per kelas di setiap split
from collections import Counter

def split_distribution(split_set, name):
    counter = Counter(label for _, label in split_set)
    df = pd.DataFrame(counter.items(), columns=['class', name]).sort_values('class')
    return df.set_index('class')

df_dist = pd.concat([
    split_distribution(train_set, 'train'),
    split_distribution(val_set, 'val'),
    split_distribution(test_set, 'test'),
], axis=1)
df_dist['total'] = df_dist.sum(axis=1)
print(df_dist.to_string())

# Plot distribusi split
fig, ax = plt.subplots(figsize=(12, 4))
df_dist[['train', 'val', 'test']].plot(kind='bar', ax=ax,
    color=['#4C72B0', '#DD8452', '#55A868'])
ax.set_title('Distribusi per Kelas setelah Split (80/10/10)')
ax.set_xlabel('Kelas'); ax.set_ylabel('Jumlah Gambar')
ax.tick_params(axis='x', rotation=45)
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('/content/split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Transformasi: Resize, Normalisasi & Augmentasi

| Stage | Transformasi |
|---|---|
| **Train** | RandomResizedCrop(224), RandomHorizontalFlip, ColorJitter, RandomRotation, ToTensor, Normalize |
| **Val/Test** | Resize(256) → CenterCrop(224), ToTensor, Normalize |

**Normalisasi:** ImageNet mean/std — standar untuk fine-tuning model pre-trained.

In [ ]:
import torchvision.transforms as T
import numpy as np

# ImageNet statistics (standar untuk food image models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Transform Definitions ─────────────────────────────────

train_transform = T.Compose([
    T.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.7, 1.0),          # random crop 70-100% area
        ratio=(0.75, 1.33)         # aspect ratio variation
    ),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.1),   # makanan kadang difoto dari atas
    T.RandomRotation(degrees=15),
    T.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3,
        hue=0.1
    ),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_test_transform = T.Compose([
    T.Resize(256),                 # resize pendek-sisi ke 256
    T.CenterCrop(IMG_SIZE),        # center crop ke 224×224
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# Denormalize helper (untuk visualisasi)
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)

print("✅ Transforms defined:")
print("  Train   :", train_transform)
print("\n  Val/Test:", val_test_transform)

## 7. PyTorch Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class VitaraFoodDataset(Dataset):
    """
    Dataset makanan untuk Vitara AI.
    Args:
        samples   : list of (Path, class_name)
        class_to_idx : dict {class_name: int}
        transform : torchvision transform
        calorie_map : dict {class_name: calories_per_100g} (opsional)
    """
    def __init__(self, samples, class_to_idx, transform=None, calorie_map=None):
        self.samples      = samples
        self.class_to_idx = class_to_idx
        self.transform    = transform
        self.calorie_map  = calorie_map or {}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, cls_name = self.samples[idx]
        img = Image.open(path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        label    = self.class_to_idx[cls_name]
        calories = self.calorie_map.get(cls_name, -1)

        return {
            'image'   : img,
            'label'   : label,
            'class'   : cls_name,
            'calories': calories,
            'path'    : str(path)
        }


# Build class index
class_to_idx = {cls: i for i, cls in enumerate(sorted(classes))}
idx_to_class = {v: k for k, v in class_to_idx.items()}
calorie_map  = dict(zip(df_cal['class_name'], df_cal['calories_per_100g']))

print("class_to_idx:")
for k, v in class_to_idx.items():
    cal = calorie_map.get(k, '?')
    print(f"  [{v:2d}] {k:<20} ({cal} kcal/100g)")

In [ ]:
BATCH_SIZE  = 32
NUM_WORKERS = 2   # Colab T4 optimal
PIN_MEMORY  = torch.cuda.is_available()

train_dataset = VitaraFoodDataset(train_set, class_to_idx, train_transform, calorie_map)
val_dataset   = VitaraFoodDataset(val_set,   class_to_idx, val_test_transform, calorie_map)
test_dataset  = VitaraFoodDataset(test_set,  class_to_idx, val_test_transform, calorie_map)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f"DataLoaders ready:")
print(f"  train : {len(train_dataset):>5} samples  →  {len(train_loader):>4} batches")
print(f"  val   : {len(val_dataset):>5} samples  →  {len(val_loader):>4} batches")
print(f"  test  : {len(test_dataset):>5} samples  →  {len(test_loader):>4} batches")
print(f"  batch size  : {BATCH_SIZE}")
print(f"  pin_memory  : {PIN_MEMORY}")

## 8. Sanity Check — Visualisasi Augmentasi

In [ ]:
def show_augmentations(dataset, n_images=4, n_aug=5):
    """
    Tampilkan n_images gambar, masing-masing dengan n_aug augmentasi berbeda.
    Membantu verifikasi bahwa augmentasi berjalan benar.
    """
    indices = random.sample(range(len(dataset)), n_images)
    fig, axes = plt.subplots(n_images, n_aug + 1, figsize=(3*(n_aug+1), 3*n_images))

    no_aug_transform = T.Compose([
        T.Resize(256), T.CenterCrop(IMG_SIZE), T.ToTensor()
    ])

    for row, idx in enumerate(indices):
        path, cls = dataset.samples[idx]
        original  = Image.open(path).convert('RGB')

        # Original (no aug)
        ax = axes[row][0]
        ax.imshow(no_aug_transform(original).permute(1, 2, 0).numpy())
        ax.set_title(f"{cls}\n(original)", fontsize=8)
        ax.axis('off')

        # Augmented versions
        for col in range(n_aug):
            aug_img = train_transform(original)
            ax = axes[row][col + 1]
            ax.imshow(denormalize(aug_img).permute(1, 2, 0).numpy())
            ax.set_title(f"aug #{col+1}", fontsize=8)
            ax.axis('off')

    plt.suptitle('Augmentasi Visualisasi — Train Set', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig('/content/augmentation_check.png', dpi=150, bbox_inches='tight')
    plt.show()

show_augmentations(train_dataset, n_images=4, n_aug=5)
print("📸 Augmentation check saved: /content/augmentation_check.png")

In [ ]:
# Cek satu batch dari train_loader
batch = next(iter(train_loader))

print("Batch shapes:")
print(f"  image    : {batch['image'].shape}   # [B, C, H, W]")
print(f"  label    : {batch['label'].shape}")
print(f"  calories : {batch['calories'][:5].tolist()} ...")
print(f"  classes  : {batch['class'][:5]}")

img_min = batch['image'].min().item()
img_max = batch['image'].max().item()
img_mean = batch['image'].mean().item()
print(f"\nPixel stats (normalized):")
print(f"  min={img_min:.3f}  max={img_max:.3f}  mean={img_mean:.3f}")
print(f"  dtype : {batch['image'].dtype}")
assert batch['image'].shape[2:] == (224, 224), "❌ Ukuran bukan 224×224!"
print("\n✅ Batch sanity check passed.")

## 9. Simpan Dataset ke Google Drive (Processed)

Pilih satu mode penyimpanan:
- **Mode A** — Simpan gambar terpreproses (JPEG) langsung ke folder `processed/`  
- **Mode B** — Simpan manifest CSV + metadata (tanpa re-copy gambar)

> Untuk dataset besar di Colab, **Mode B lebih cepat dan hemat storage** karena menghindari duplikasi file. Mode A berguna jika pipeline inference perlu gambar standalone.

In [ ]:
# ── PILIH MODE ────────────────────────────
SAVE_MODE = "B"  # "A" = copy gambar | "B" = manifest CSV saja
# ─────────────────────────────────────────

In [ ]:
import shutil
from tqdm.auto import tqdm

os.makedirs(PROCESSED_DIR, exist_ok=True)

def build_manifest(split_name, split_data, class_to_idx, calorie_map):
    rows = []
    for path, cls in split_data:
        rows.append({
            'split'           : split_name,
            'original_path'   : str(path),
            'filename'        : path.name,
            'class_name'      : cls,
            'label_idx'       : class_to_idx[cls],
            'calories_per_100g': calorie_map.get(cls, None)
        })
    return pd.DataFrame(rows)


if SAVE_MODE == "A":
    # ── Mode A: copy & resize gambar ke processed folder ──
    print("Mode A: Menyalin gambar ke PROCESSED_DIR...")

    save_transform = T.Compose([T.Resize(256), T.CenterCrop(IMG_SIZE)])

    splits_data = {
        'train': train_set,
        'val'  : val_set,
        'test' : test_set
    }

    for split_name, split_data in splits_data.items():
        print(f"  Processing {split_name} ({len(split_data)} gambar)...")
        for path, cls in tqdm(split_data, desc=split_name):
            dst_dir = os.path.join(PROCESSED_DIR, split_name, cls)
            os.makedirs(dst_dir, exist_ok=True)
            dst_path = os.path.join(dst_dir, path.name)
            if not os.path.exists(dst_path):
                img = Image.open(path).convert('RGB')
                img = save_transform(img)
                img.save(dst_path, format='JPEG', quality=95)

    print(f"\n✅ Mode A selesai. Gambar tersimpan di: {PROCESSED_DIR}")

else:
    # ── Mode B: simpan manifest CSV saja ──────────────────
    print("Mode B: Menyimpan manifest CSV...")

    dfs = []
    for split_name, split_data in [('train', train_set), ('val', val_set), ('test', test_set)]:
        df = build_manifest(split_name, split_data, class_to_idx, calorie_map)
        dfs.append(df)

    df_manifest = pd.concat(dfs, ignore_index=True)

    # Simpan manifest lengkap
    manifest_path = os.path.join(PROCESSED_DIR, 'manifest.csv')
    df_manifest.to_csv(manifest_path, index=False)
    print(f"  ✅ manifest.csv saved → {manifest_path}")

    # Simpan split terpisah
    for split_name in ['train', 'val', 'test']:
        sub = df_manifest[df_manifest['split'] == split_name]
        p = os.path.join(PROCESSED_DIR, f'{split_name}.csv')
        sub.to_csv(p, index=False)
        print(f"  ✅ {split_name}.csv saved → {p}  ({len(sub)} rows)")

print(f"\n📁 Output di Google Drive: {PROCESSED_DIR}")

## 10. Simpan class_to_idx & Metadata Preprocessing

In [ ]:
import json

metadata = {
    "pipeline_version"  : "1.0.0",
    "dataset_contract"  : "Vitara AI — Vision Food Images",
    "created_by"        : "Bagus",
    "img_size"          : IMG_SIZE,
    "normalize_mean"    : IMAGENET_MEAN,
    "normalize_std"     : IMAGENET_STD,
    "split_ratio"       : {
        "train": TRAIN_RATIO,
        "val"  : VAL_RATIO,
        "test" : TEST_RATIO
    },
    "random_seed"       : RANDOM_SEED,
    "num_classes"       : len(classes),
    "class_to_idx"      : class_to_idx,
    "idx_to_class"      : {str(k): v for k, v in idx_to_class.items()},
    "calorie_map"       : calorie_map,
    "split_counts"      : {
        "train": len(train_set),
        "val"  : len(val_set),
        "test" : len(test_set)
    },
    "train_augmentations": [
        "RandomResizedCrop(224, scale=(0.7,1.0))",
        "RandomHorizontalFlip(p=0.5)",
        "RandomVerticalFlip(p=0.1)",
        "RandomRotation(15)",
        "ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1)",
        "RandomGrayscale(p=0.05)",
        "Normalize(ImageNet)"
    ],
    "val_test_transforms": [
        "Resize(256)",
        "CenterCrop(224)",
        "Normalize(ImageNet)"
    ],
    "save_mode": SAVE_MODE
}

meta_path = os.path.join(PROCESSED_DIR, 'preprocessing_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✅ Metadata saved → {meta_path}")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

## 11. GPU Benchmark — DataLoader Speed Test

In [ ]:
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

N_BATCHES = min(20, len(train_loader))   # test 20 batch pertama
total_images = 0
start = time.time()

for i, batch in enumerate(train_loader):
    if i >= N_BATCHES:
        break
    imgs   = batch['image'].to(device, non_blocking=True)
    labels = batch['label'].to(device, non_blocking=True)
    total_images += imgs.size(0)

elapsed = time.time() - start
throughput = total_images / elapsed

print(f"\n⚡ DataLoader Benchmark ({N_BATCHES} batches):")
print(f"  Total images  : {total_images}")
print(f"  Elapsed       : {elapsed:.2f}s")
print(f"  Throughput    : {throughput:.1f} images/sec")

if torch.cuda.is_available():
    mem_alloc = torch.cuda.memory_allocated() / 1e6
    mem_reserved = torch.cuda.memory_reserved() / 1e6
    print(f"  GPU VRAM alloc: {mem_alloc:.1f} MB")
    print(f"  GPU VRAM rsrv : {mem_reserved:.1f} MB")

## 12. Ringkasan Final

In [ ]:
print("="*60)
print("✅  VITARA AI — VISION PREPROCESSING SELESAI")
print("="*60)
print(f"""
Dataset Contract Compliance:
  ✅  Format JPEG/PNG
  ✅  Min 224×224 px (resize & crop)
  ✅  calorie_map.csv tervalidasi
  ✅  classes.txt tervalidasi
  ✅  Duplikat antar kelas dicek

Split Summary:
  Train : {len(train_set):>6} gambar  (80%)
  Val   : {len(val_set):>6} gambar  (10%)
  Test  : {len(test_set):>6} gambar  (10%)
  Kelas : {len(classes)}

Image Processing:
  Resize target      : 224×224
  Normalisasi        : ImageNet mean/std
  Augmentasi (train) : RandomResizedCrop, Flip,
                       Rotation, ColorJitter, Grayscale

Output (Google Drive):
  {PROCESSED_DIR}/
    ├── manifest.csv
    ├── train.csv
    ├── val.csv
    ├── test.csv
    └── preprocessing_metadata.json
""")
print("="*60)
print("Next step: Training model → model_training.ipynb")